In [1]:
from pathlib import Path
from collections import Counter
import pandas as pd
import json
import re

OUT_DIR = Path("../seed_exports")
EDA_DIR = OUT_DIR / "EDA"

CLEAN_JSONL = OUT_DIR / "wiki_mcq_seed_clean_new.jsonl"
MATH_ISSUES_CSV = EDA_DIR / "wiki_mcq_math_format_issues.csv"

OUT_FIXED_OR_KEPT_JSONL = OUT_DIR / "wiki_mcq_seed_clean_math_triaged.jsonl"
OUT_DROPPED_MATH_JSONL = OUT_DIR / "wiki_mcq_seed_clean_math_dropped.jsonl"
OUT_MATH_TRIAGE_REPORT_JSON = EDA_DIR / "wiki_mcq_math_triage_report.json"

In [2]:
def read_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

def write_jsonl(path, rows):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

clean_rows = read_jsonl(CLEAN_JSONL)
df_issues = pd.read_csv(MATH_ISSUES_CSV)
df_issues["source_id"] = df_issues["source_id"].astype(str)

issue_source_ids = set(df_issues["source_id"])

print("Clean rows:", len(clean_rows))
print("Math issue rows:", len(issue_source_ids))
print("Issue ratio:", round(len(issue_source_ids) / len(clean_rows) * 100, 2), "%")

Clean rows: 7946
Math issue rows: 399
Issue ratio: 5.02 %


### Improved scanner để giảm false positive

In [3]:
def clean_text(x):
    if x is None:
        return ""
    x = str(x)
    x = x.replace("\u00a0", " ")
    x = re.sub(r"\s+", " ", x).strip()
    return x


def is_valid_user_content(text):
    if not isinstance(text, str):
        return False

    required_patterns = [
        r"^Câu hỏi:",
        r"\nA\.",
        r"\nB\.",
        r"\nC\.",
        r"\nD\.",
    ]

    return all(re.search(p, text) for p in required_patterns)


CURRENCY_DOLLAR_PATTERN = re.compile(
    r"\$\s?\d{1,3}(?:,\d{3})+(?:\.\d+)?|\$\s?\d+\.\d{2}"
)

LATEX_FRAC_PATTERN = re.compile(r"\\frac\{[^{}\n]+\}\{[^{}\n]+\}")
LATEX_SQRT_PATTERN = re.compile(r"\\sqrt\{[^{}\n]+\}")
POWER_PATTERN = re.compile(
    r"(?<![\w$])(?:\d+|[a-zA-Z])\^\{[^{}\n]+\}(?![\w$])"
    r"|(?<![\w$])(?:\d+|[a-zA-Z])\^\d+(?![\w$])"
)


def math_dollar_positions(text):
    """
    Dollar delimiter positions, excluding obvious currency like $35,095.
    """
    ignored_positions = {
        m.start()
        for m in CURRENCY_DOLLAR_PATTERN.finditer(text)
    }

    return [
        m.start()
        for m in re.finditer(r"\$", text)
        if m.start() not in ignored_positions
    ]


def is_inside_math_dollars(text, pos):
    positions = math_dollar_positions(text)
    num_before = sum(1 for p in positions if p < pos)
    return num_before % 2 == 1


def has_unbalanced_math_dollars(text):
    return len(math_dollar_positions(text)) % 2 != 0


def detect_true_math_format_flags(text):
    flags = []

    if not is_valid_user_content(text):
        flags.append("invalid_user_content_format")

    if has_unbalanced_math_dollars(text):
        flags.append("unbalanced_math_dollar")

    for m in LATEX_FRAC_PATTERN.finditer(text):
        if not is_inside_math_dollars(text, m.start()):
            flags.append("raw_latex_frac")
            break

    for m in LATEX_SQRT_PATTERN.finditer(text):
        if not is_inside_math_dollars(text, m.start()):
            flags.append("raw_latex_sqrt")
            break

    for m in POWER_PATTERN.finditer(text):
        if not is_inside_math_dollars(text, m.start()):
            flags.append("raw_caret_power")
            break

    if "\\n" in text:
        flags.append("literal_backslash_n")

    if re.search(r"&[a-zA-Z]+;", text):
        flags.append("html_entity")

    if re.search(r"\{\s*\}|\[\s*\]", text):
        flags.append("broken_placeholder")

    if "\\begin" in text:
        flags.append("raw_latex_begin")

    if "\\end" in text:
        flags.append("raw_latex_end")

    return sorted(set(flags))

### Rule-based fix cho lỗi đơn giản

In [4]:
def wrap_matches_outside_math(text, pattern):
    result = []
    last = 0
    changed = False

    for m in pattern.finditer(text):
        start, end = m.span()
        expr = m.group(0)

        if is_inside_math_dollars(text, start):
            continue

        result.append(text[last:start])
        result.append(f"${expr}$")
        last = end
        changed = True

    if not changed:
        return text

    result.append(text[last:])
    return "".join(result)


def deterministic_math_fix(text):
    fixed = text
    fixed = wrap_matches_outside_math(fixed, LATEX_FRAC_PATTERN)
    fixed = wrap_matches_outside_math(fixed, LATEX_SQRT_PATTERN)
    fixed = wrap_matches_outside_math(fixed, POWER_PATTERN)
    return fixed

#### Triage 399 issue rows

In [5]:
source_id_to_row = {
    str(row["metadata"]["source_id"]): row
    for row in clean_rows
}

kept_rows = []
dropped_rows = []

triage_log = []

for row in clean_rows:
    source_id = str(row["metadata"]["source_id"])

    # Không nằm trong issue CSV thì giữ nguyên.
    if source_id not in issue_source_ids:
        kept_rows.append(row)
        continue

    row_new = dict(row)
    row_new["metadata"] = dict(row["metadata"])

    original_text = row_new["messages"][0]["content"]

    before_flags = detect_true_math_format_flags(original_text)

    # Nếu scanner cải tiến thấy không còn lỗi thật, coi là false positive và giữ.
    if not before_flags:
        row_new["metadata"].setdefault("qc_flags", [])
        row_new["metadata"]["qc_flags"].append("math_format_issue_false_positive")
        row_new["metadata"]["math_format_triage"] = {
            "action": "keep_false_positive",
            "before_flags": before_flags,
        }
        kept_rows.append(row_new)

        triage_log.append({
            "source_id": source_id,
            "action": "keep_false_positive",
            "before_flags": before_flags,
            "after_flags": [],
        })
        continue

    # Thử rule-based fix.
    fixed_text = deterministic_math_fix(original_text)
    after_flags = detect_true_math_format_flags(fixed_text)

    if not after_flags:
        row_new["messages"][0]["content"] = fixed_text
        row_new["metadata"].setdefault("qc_flags", [])
        row_new["metadata"]["qc_flags"].append("math_format_fixed_by_rule")
        row_new["metadata"]["math_format_triage"] = {
            "action": "fixed_by_rule",
            "before_flags": before_flags,
            "after_flags": after_flags,
        }
        kept_rows.append(row_new)

        triage_log.append({
            "source_id": source_id,
            "action": "fixed_by_rule",
            "before_flags": before_flags,
            "after_flags": after_flags,
        })
        continue

    # Nếu vẫn còn lỗi sau rule-fix thì drop.
    row_new["metadata"].setdefault("qc_flags", [])
    row_new["metadata"]["qc_flags"].append("dropped_math_format_issue")
    row_new["metadata"]["qc_action"] = "drop_math_format_issue"
    row_new["metadata"]["math_format_triage"] = {
        "action": "drop_unresolved_math_format_issue",
        "before_flags": before_flags,
        "after_flags": after_flags,
    }

    dropped_rows.append(row_new)

    triage_log.append({
        "source_id": source_id,
        "action": "drop_unresolved_math_format_issue",
        "before_flags": before_flags,
        "after_flags": after_flags,
    })


print("Original clean rows:", len(clean_rows))
print("Rows kept after triage:", len(kept_rows))
print("Rows dropped after triage:", len(dropped_rows))
print("Final drop ratio:", round(len(dropped_rows) / len(clean_rows) * 100, 2), "%")

Original clean rows: 7946
Rows kept after triage: 7929
Rows dropped after triage: 17
Final drop ratio: 0.21 %


### Sumary

In [6]:
action_counter = Counter(x["action"] for x in triage_log)

print("Triage action counts:")
display(pd.Series(action_counter).sort_values(ascending=False))

before_flag_counter = Counter()
after_flag_counter = Counter()

for x in triage_log:
    for flag in x["before_flags"]:
        before_flag_counter[flag] += 1
    for flag in x["after_flags"]:
        after_flag_counter[flag] += 1

print("\nBefore true flags:")
display(pd.Series(before_flag_counter).sort_values(ascending=False))

print("\nRemaining flags in dropped rows:")
display(pd.Series(after_flag_counter).sort_values(ascending=False))

Triage action counts:


keep_false_positive                  252
fixed_by_rule                        130
drop_unresolved_math_format_issue     17
dtype: int64


Before true flags:


raw_caret_power           74
raw_latex_frac            50
unbalanced_math_dollar    17
raw_latex_sqrt            12
dtype: int64


Remaining flags in dropped rows:


unbalanced_math_dollar    17
dtype: int64

### Subject impact

In [7]:
before_subject_counter = Counter(
    row["metadata"].get("subject")
    for row in clean_rows
)

after_subject_counter = Counter(
    row["metadata"].get("subject")
    for row in kept_rows
)

dropped_subject_counter = Counter(
    row["metadata"].get("subject")
    for row in dropped_rows
)

df_subject_impact = pd.DataFrame([
    {
        "subject": subject,
        "before": before_subject_counter[subject],
        "after": after_subject_counter.get(subject, 0),
        "dropped": dropped_subject_counter.get(subject, 0),
        "drop_percent_within_subject": round(
            dropped_subject_counter.get(subject, 0) / before_subject_counter[subject] * 100,
            2
        ),
    }
    for subject in sorted(before_subject_counter)
])

df_subject_impact = df_subject_impact.sort_values(
    ["dropped", "drop_percent_within_subject"],
    ascending=False
)

display(df_subject_impact.head(30))

,subject,before,after,dropped,drop_percent_within_subject
7,elementary_mathematics,418,405,13,3.11
16,high_school_statistics,243,240,3,1.23
13,high_school_mathematics,302,301,1,0.33
0,abstract_algebra,116,116,0,0.00
1,anatomy,152,152,0,0.00
2,astronomy,173,173,0,0.00
3,business_ethics,114,114,0,0.00
4,college_biology,164,164,0,0.00
5,computer_security,116,116,0,0.00
6,conceptual_physics,264,264,0,0.00


### Answer distribution impact

In [8]:
before_answer_counter = Counter(
    row["metadata"].get("answer")
    for row in clean_rows
)

after_answer_counter = Counter(
    row["metadata"].get("answer")
    for row in kept_rows
)

dropped_answer_counter = Counter(
    row["metadata"].get("answer")
    for row in dropped_rows
)

print("Before answer distribution:")
display(pd.Series(before_answer_counter).sort_index())

print("After answer distribution:")
display(pd.Series(after_answer_counter).sort_index())

print("Dropped answer distribution:")
display(pd.Series(dropped_answer_counter).sort_index())

Before answer distribution:


A    1820
B    2000
C    2058
D    2068
dtype: int64

After answer distribution:


A    1819
B    1994
C    2053
D    2063
dtype: int64

Dropped answer distribution:


A    1
B    6
C    5
D    5
dtype: int64

### Export

In [9]:
write_jsonl(OUT_FIXED_OR_KEPT_JSONL, kept_rows)
write_jsonl(OUT_DROPPED_MATH_JSONL, dropped_rows)

triage_df = pd.DataFrame(triage_log)

report = {
    "input_clean_jsonl": str(CLEAN_JSONL),
    "math_issues_csv": str(MATH_ISSUES_CSV),
    "output_jsonl": str(OUT_FIXED_OR_KEPT_JSONL),
    "dropped_jsonl": str(OUT_DROPPED_MATH_JSONL),
    "input_rows": int(len(clean_rows)),
    "math_issue_rows_from_csv": int(len(issue_source_ids)),
    "kept_rows": int(len(kept_rows)),
    "dropped_rows": int(len(dropped_rows)),
    "drop_ratio_percent": round(len(dropped_rows) / len(clean_rows) * 100, 2),
    "triage_action_counts": {
        str(k): int(v)
        for k, v in action_counter.items()
    },
    "before_true_flag_counts": {
        str(k): int(v)
        for k, v in before_flag_counter.items()
    },
    "remaining_flag_counts_in_dropped": {
        str(k): int(v)
        for k, v in after_flag_counter.items()
    },
    "answer_distribution_before": {
        str(k): int(v)
        for k, v in sorted(before_answer_counter.items())
    },
    "answer_distribution_after": {
        str(k): int(v)
        for k, v in sorted(after_answer_counter.items())
    },
    "dropped_subject_distribution": {
        str(k): int(v)
        for k, v in pd.Series(dropped_subject_counter).sort_values(ascending=False).items()
    },
}

with open(OUT_MATH_TRIAGE_REPORT_JSON, "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=2)

print(json.dumps(report, ensure_ascii=False, indent=2))

{
  "input_clean_jsonl": "../seed_exports/wiki_mcq_seed_clean_new.jsonl",
  "math_issues_csv": "../seed_exports/EDA/wiki_mcq_math_format_issues.csv",
  "output_jsonl": "../seed_exports/wiki_mcq_seed_clean_math_triaged.jsonl",
  "dropped_jsonl": "../seed_exports/wiki_mcq_seed_clean_math_dropped.jsonl",
  "input_rows": 7946,
  "math_issue_rows_from_csv": 399,
  "kept_rows": 7929,
  "dropped_rows": 17,
  "drop_ratio_percent": 0.21,
  "triage_action_counts": {
    "keep_false_positive": 252,
    "fixed_by_rule": 130,
    "drop_unresolved_math_format_issue": 17
  },
  "before_true_flag_counts": {
    "raw_latex_frac": 50,
    "raw_caret_power": 74,
    "raw_latex_sqrt": 12,
    "unbalanced_math_dollar": 17
  },
  "remaining_flag_counts_in_dropped": {
    "unbalanced_math_dollar": 17
  },
  "answer_distribution_before": {
    "A": 1820,
    "B": 2000,
    "C": 2058,
    "D": 2068
  },
  "answer_distribution_after": {
    "A": 1819,
    "B": 1994,
    "C": 2053,
    "D": 2063
  },
  "dropped_

### Chốt file triaged làm bản final

In [11]:
from pathlib import Path
import shutil

src = Path("../seed_exports/wiki_mcq_seed_clean_math_triaged.jsonl")
dst = Path("../seed_exports/wiki_mcq_seed_final.jsonl")

shutil.copyfile(src, dst)

print("Final wiki_mcq seed:", dst)

Final wiki_mcq seed: ../seed_exports/wiki_mcq_seed_final.jsonl


In [14]:
FINAL_JSONL = Path("../seed_exports/wiki_mcq_seed_final.jsonl")

In [15]:
rows = read_jsonl(FINAL_JSONL)

print("File:", FINAL_JSONL)
print("Total rows:", len(rows))

File: ../seed_exports/wiki_mcq_seed_final.jsonl
Total rows: 7929


In [16]:
WIKI_MCQ_KEEP_SUBJECTS = {
    # General / misc
    "miscellaneous",
    "world_religions",
    "sociology",
    "philosophy",
    "prehistory",
    "logical_fallacies",

    # History / geography
    "high_school_world_history",
    "high_school_european_history",
    "high_school_geography",

    # Science phổ thông
    "astronomy",
    "conceptual_physics",
    "high_school_physics",
    "high_school_chemistry",
    "high_school_biology",
    "college_biology",
    "virology",
    "anatomy",
    "nutrition",
    "human_aging",
    "medical_genetics",

    # Math / statistics
    "elementary_mathematics",
    "high_school_mathematics",
    "high_school_statistics",
    "abstract_algebra",

    # Economics / business / technology mức general
    "high_school_macroeconomics",
    "high_school_microeconomics",
    "marketing",
    "management",
    "business_ethics",
    "public_relations",
    "machine_learning",
    "computer_security",
    "security_studies",
}

In [17]:
ANSWER_LETTERS = {"A", "B", "C", "D"}


def clean_text(x):
    if x is None:
        return ""

    x = str(x)
    x = x.replace("\u00a0", " ")
    x = re.sub(r"\s+", " ", x).strip()
    return x


def is_valid_assistant_answer(item):
    try:
        content = item["messages"][1].get("content", "")
    except Exception:
        return False

    return bool(re.match(r"^Đáp án:\s*[ABCD]$", content.strip()))


def get_assistant_answer(item):
    try:
        content = item["messages"][1].get("content", "").strip()
    except Exception:
        return None

    m = re.match(r"^Đáp án:\s*([ABCD])$", content)
    return m.group(1) if m else None


def parse_user_content(user_content):
    """
    Parse format:
    Câu hỏi: ...
    A. ...
    B. ...
    C. ...
    D. ...

    Supports wrapped option lines.
    """
    if not isinstance(user_content, str):
        return None

    lines = user_content.splitlines()

    question_lines = []
    choices = {}
    current_choice = None

    for line in lines:
        raw_line = line.rstrip()
        stripped = raw_line.strip()

        m = re.match(r"^([ABCD])\.\s*(.*)$", stripped)

        if m:
            current_choice = m.group(1)
            choices[current_choice] = m.group(2).strip()
            continue

        if current_choice is not None:
            choices[current_choice] += " " + stripped
        else:
            question_lines.append(stripped)

    question = "\n".join(question_lines).strip()
    question = re.sub(r"^Câu hỏi:\s*", "", question).strip()

    choices = {
        k: clean_text(v)
        for k, v in choices.items()
    }

    if set(choices.keys()) != ANSWER_LETTERS:
        return None

    return {
        "question": clean_text(question),
        "choices": choices,
    }


def has_duplicate_choices(item):
    try:
        user_content = item["messages"][0]["content"]
    except Exception:
        return True

    parsed = parse_user_content(user_content)

    if parsed is None:
        return True

    choices = parsed["choices"]

    normalized = [
        re.sub(r"\s+", " ", choices[label].lower().strip())
        for label in ["A", "B", "C", "D"]
    ]

    return len(set(normalized)) < 4

### Math format scanner

In [18]:
CURRENCY_DOLLAR_PATTERN = re.compile(
    r"\$\s?\d{1,3}(?:,\d{3})+(?:\.\d+)?|\$\s?\d+\.\d{2}"
)

LATEX_FRAC_PATTERN = re.compile(r"\\frac\{[^{}\n]+\}\{[^{}\n]+\}")
LATEX_SQRT_PATTERN = re.compile(r"\\sqrt\{[^{}\n]+\}")
POWER_PATTERN = re.compile(
    r"(?<![\w$])(?:\d+|[a-zA-Z])\^\{[^{}\n]+\}(?![\w$])"
    r"|(?<![\w$])(?:\d+|[a-zA-Z])\^\d+(?![\w$])"
)


def math_dollar_positions(text):
    """
    Dollar delimiter positions, excluding obvious currency like $35,095.
    """
    ignored_positions = {
        m.start()
        for m in CURRENCY_DOLLAR_PATTERN.finditer(text)
    }

    return [
        m.start()
        for m in re.finditer(r"\$", text)
        if m.start() not in ignored_positions
    ]


def is_inside_math_dollars(text, pos):
    positions = math_dollar_positions(text)
    num_before = sum(1 for p in positions if p < pos)
    return num_before % 2 == 1


def has_unbalanced_math_dollars(text):
    return len(math_dollar_positions(text)) % 2 != 0


def is_valid_user_content(text):
    if not isinstance(text, str):
        return False

    required_patterns = [
        r"^Câu hỏi:",
        r"\nA\.",
        r"\nB\.",
        r"\nC\.",
        r"\nD\.",
    ]

    return all(re.search(p, text) for p in required_patterns)


def detect_true_math_format_flags(text):
    flags = []

    if not is_valid_user_content(text):
        flags.append("invalid_user_content_format")

    if has_unbalanced_math_dollars(text):
        flags.append("unbalanced_math_dollar")

    for m in LATEX_FRAC_PATTERN.finditer(text):
        if not is_inside_math_dollars(text, m.start()):
            flags.append("raw_latex_frac")
            break

    for m in LATEX_SQRT_PATTERN.finditer(text):
        if not is_inside_math_dollars(text, m.start()):
            flags.append("raw_latex_sqrt")
            break

    for m in POWER_PATTERN.finditer(text):
        if not is_inside_math_dollars(text, m.start()):
            flags.append("raw_caret_power")
            break

    if "\\n" in text:
        flags.append("literal_backslash_n")

    if re.search(r"&[a-zA-Z]+;", text):
        flags.append("html_entity")

    if re.search(r"\{\s*\}|\[\s*\]", text):
        flags.append("broken_placeholder")

    if "\\begin" in text:
        flags.append("raw_latex_begin")

    if "\\end" in text:
        flags.append("raw_latex_end")

    return sorted(set(flags))

In [19]:
schema_errors = []
assistant_format_errors = []
duplicate_choice_errors = []
subject_whitelist_errors = []
math_format_errors = []

answer_counter = Counter()
subject_counter = Counter()
qc_action_counter = Counter()
qc_flag_counter = Counter()

source_id_counter = Counter()

for idx, item in enumerate(rows):
    # ---------------------------------------------------------
    # Basic schema
    # ---------------------------------------------------------
    if not isinstance(item, dict):
        schema_errors.append((idx, "item_not_dict"))
        continue

    if "messages" not in item:
        schema_errors.append((idx, "missing_messages"))
        continue

    if "metadata" not in item:
        schema_errors.append((idx, "missing_metadata"))
        continue

    messages = item["messages"]
    metadata = item["metadata"]

    if not isinstance(messages, list) or len(messages) != 2:
        schema_errors.append((idx, "invalid_messages_length"))
        continue

    if not isinstance(messages[0], dict) or not isinstance(messages[1], dict):
        schema_errors.append((idx, "message_not_dict"))
        continue

    if messages[0].get("role") != "user":
        schema_errors.append((idx, "invalid_user_role"))

    if messages[1].get("role") != "assistant":
        schema_errors.append((idx, "invalid_assistant_role"))

    user_content = messages[0].get("content")
    assistant_content = messages[1].get("content")

    if not isinstance(user_content, str) or not user_content.strip():
        schema_errors.append((idx, "empty_user_content"))

    if not isinstance(assistant_content, str) or not assistant_content.strip():
        schema_errors.append((idx, "empty_assistant_content"))

    # ---------------------------------------------------------
    # Assistant answer format
    # ---------------------------------------------------------
    if not is_valid_assistant_answer(item):
        assistant_format_errors.append((idx, metadata.get("source_id"), assistant_content))
    else:
        answer = get_assistant_answer(item)
        answer_counter[answer] += 1

    # ---------------------------------------------------------
    # Duplicate choices
    # ---------------------------------------------------------
    if has_duplicate_choices(item):
        duplicate_choice_errors.append((idx, metadata.get("source_id")))

    # ---------------------------------------------------------
    # Subject whitelist
    # ---------------------------------------------------------
    subject = metadata.get("subject")
    subject_counter[subject] += 1

    if subject not in WIKI_MCQ_KEEP_SUBJECTS:
        subject_whitelist_errors.append((idx, metadata.get("source_id"), subject))

    # ---------------------------------------------------------
    # Math format issue
    # ---------------------------------------------------------
    flags = detect_true_math_format_flags(user_content)

    if flags:
        math_format_errors.append({
            "row_idx": idx,
            "source_id": metadata.get("source_id"),
            "subject": subject,
            "answer": get_assistant_answer(item),
            "flags": flags,
            "text": user_content,
        })

    # ---------------------------------------------------------
    # Metadata counters
    # ---------------------------------------------------------
    source_id_counter[str(metadata.get("source_id"))] += 1

    qc_action_counter[metadata.get("qc_action")] += 1

    for flag in metadata.get("qc_flags", []):
        qc_flag_counter[flag] += 1

In [20]:
print("FINAL WIKI_MCQ SEED RECHECK")
print("File:", FINAL_JSONL)
print("Total rows:", len(rows))

print("\nSchema messages errors:", len(schema_errors))
print("Assistant answer format errors:", len(assistant_format_errors))
print("Duplicate choice errors:", len(duplicate_choice_errors))
print("Subject whitelist errors:", len(subject_whitelist_errors))
print("Remaining math format errors:", len(math_format_errors))

source_id_duplicates = {
    source_id: count
    for source_id, count in source_id_counter.items()
    if count > 1
}

print("Duplicate source_id:", len(source_id_duplicates))

print("\nAnswer distribution:")
display(pd.Series(answer_counter).sort_index())

print("\nSubject distribution:")
display(pd.Series(subject_counter).sort_values(ascending=False))

print("\nQC action distribution:")
display(pd.Series(qc_action_counter).sort_values(ascending=False))

print("\nQC flag distribution:")
display(pd.Series(qc_flag_counter).sort_values(ascending=False))

FINAL WIKI_MCQ SEED RECHECK
File: ../seed_exports/wiki_mcq_seed_final.jsonl
Total rows: 7929

Schema messages errors: 0
Assistant answer format errors: 0
Duplicate choice errors: 0
Subject whitelist errors: 0
Remaining math format errors: 29
Duplicate source_id: 0

Answer distribution:


A    1819
B    1994
C    2053
D    2063
dtype: int64


Subject distribution:


miscellaneous                   864
high_school_macroeconomics      437
elementary_mathematics          405
prehistory                      358
philosophy                      346
nutrition                       342
high_school_biology             341
high_school_mathematics         301
high_school_microeconomics      267
conceptual_physics              264
marketing                       264
human_aging                     248
high_school_statistics          240
high_school_chemistry           228
high_school_geography           225
sociology                       223
security_studies                211
world_religions                 194
high_school_world_history       189
virology                        187
logical_fallacies               185
astronomy                       173
high_school_physics             171
college_biology                 164
anatomy                         152
machine_learning                128
public_relations                123
management                  


QC action distribution:


keep    7929
dtype: int64


QC flag distribution:


math_format_issue_false_positive    252
math_format_fixed_by_rule           130
dtype: int64

In [21]:
print("Schema errors sample:")
display(pd.DataFrame(schema_errors[:20], columns=["row_idx", "error"]))

print("Assistant format errors sample:")
display(pd.DataFrame(
    assistant_format_errors[:20],
    columns=["row_idx", "source_id", "assistant_content"]
))

print("Duplicate choice errors sample:")
display(pd.DataFrame(
    duplicate_choice_errors[:20],
    columns=["row_idx", "source_id"]
))

print("Subject whitelist errors sample:")
display(pd.DataFrame(
    subject_whitelist_errors[:20],
    columns=["row_idx", "source_id", "subject"]
))

print("Math format errors sample:")
display(pd.DataFrame(math_format_errors[:20]))

Schema errors sample:


,row_idx,error


Assistant format errors sample:


,row_idx,source_id,assistant_content


Duplicate choice errors sample:


,row_idx,source_id


Subject whitelist errors sample:


,row_idx,source_id,subject


Math format errors sample:


,row_idx,source_id,subject,answer,flags,text
0,44,machine_learning/test/107,machine_learning,B,"[raw_latex_begin, raw_latex_end]",Câu hỏi: Câu nào sau đây đúng về hạt nhân tích...
1,529,high_school_european_history/test/118,high_school_european_history,B,[literal_backslash_n],Câu hỏi: Câu hỏi này liên quan đến thông tin s...
2,1022,high_school_european_history/test/83,high_school_european_history,A,[html_entity],Câu hỏi: Câu hỏi này liên quan đến thông tin s...
3,1114,machine_learning/test/106,machine_learning,C,[literal_backslash_n],Câu hỏi: Giả sử chúng ta có hàm mục tiêu sau: ...
4,1735,high_school_macroeconomics/test/316,high_school_macroeconomics,C,[unbalanced_math_dollar],Câu hỏi: Cái nào sau đây sẽ có tác động đến GD...
5,2066,high_school_macroeconomics/test/273,high_school_macroeconomics,C,[unbalanced_math_dollar],Câu hỏi: Hãy tưởng tượng một nền kinh tế chỉ s...
6,2347,miscellaneous/test/310,miscellaneous,A,[raw_caret_power],Câu hỏi: Mỗi năm ở Hoa Kỳ sản xuất bao nhiêu k...
7,2528,high_school_microeconomics/test/44,high_school_microeconomics,B,[unbalanced_math_dollar],Câu hỏi: Cái nào sau đây là ví dụ tốt nhất về ...
8,2575,high_school_macroeconomics/test/150,high_school_macroeconomics,C,[unbalanced_math_dollar],Câu hỏi: Nền kinh tế Hoa Kỳ hiện đang gặp khó ...
9,2587,high_school_microeconomics/test/32,high_school_microeconomics,D,[unbalanced_math_dollar],Câu hỏi: Jason làm sạch hồ bơi trong một thị t...


In [22]:
FINAL_RECHECK_REPORT_JSON = Path("seed_exports/EDA/wiki_mcq_seed_final_recheck_report.json")
FINAL_MATH_ERRORS_CSV = Path("seed_exports/EDA/wiki_mcq_seed_final_remaining_math_errors.csv")

FINAL_RECHECK_REPORT_JSON.parent.mkdir(parents=True, exist_ok=True)

df_math_errors = pd.DataFrame(math_format_errors)
df_math_errors.to_csv(FINAL_MATH_ERRORS_CSV, index=False)

report = {
    "file": str(FINAL_JSONL),
    "total_rows": int(len(rows)),

    "schema_messages_errors": int(len(schema_errors)),
    "assistant_answer_format_errors": int(len(assistant_format_errors)),
    "duplicate_choice_errors": int(len(duplicate_choice_errors)),
    "subject_whitelist_errors": int(len(subject_whitelist_errors)),
    "remaining_math_format_errors": int(len(math_format_errors)),
    "duplicate_source_id_count": int(len(source_id_duplicates)),

    "answer_distribution": {
        str(k): int(v)
        for k, v in sorted(answer_counter.items())
    },

    "subject_distribution": {
        str(k): int(v)
        for k, v in pd.Series(subject_counter).sort_values(ascending=False).items()
    },

    "qc_action_distribution": {
        str(k): int(v)
        for k, v in pd.Series(qc_action_counter).sort_values(ascending=False).items()
    },

    "qc_flag_distribution": {
        str(k): int(v)
        for k, v in pd.Series(qc_flag_counter).sort_values(ascending=False).items()
    },

    "remaining_math_errors_csv": str(FINAL_MATH_ERRORS_CSV),
}

with open(FINAL_RECHECK_REPORT_JSON, "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=2)

print(json.dumps(report, ensure_ascii=False, indent=2))

{
  "file": "../seed_exports/wiki_mcq_seed_final.jsonl",
  "total_rows": 7929,
  "schema_messages_errors": 0,
  "assistant_answer_format_errors": 0,
  "duplicate_choice_errors": 0,
  "subject_whitelist_errors": 0,
  "remaining_math_format_errors": 29,
  "duplicate_source_id_count": 0,
  "answer_distribution": {
    "A": 1819,
    "B": 1994,
    "C": 2053,
    "D": 2063
  },
  "subject_distribution": {
    "miscellaneous": 864,
    "high_school_macroeconomics": 437,
    "elementary_mathematics": 405,
    "prehistory": 358,
    "philosophy": 346,
    "nutrition": 342,
    "high_school_biology": 341,
    "high_school_mathematics": 301,
    "high_school_microeconomics": 267,
    "conceptual_physics": 264,
    "marketing": 264,
    "human_aging": 248,
    "high_school_statistics": 240,
    "high_school_chemistry": 228,
    "high_school_geography": 225,
    "sociology": 223,
    "security_studies": 211,
    "world_religions": 194,
    "high_school_world_history": 189,
    "virology": 187,
  

In [23]:
assert len(rows) > 0, "Final file is empty."

assert len(schema_errors) == 0, f"Schema errors found: {len(schema_errors)}"
assert len(assistant_format_errors) == 0, f"Assistant format errors found: {len(assistant_format_errors)}"
assert len(duplicate_choice_errors) == 0, f"Duplicate choices found: {len(duplicate_choice_errors)}"
assert len(subject_whitelist_errors) == 0, f"Subject whitelist errors found: {len(subject_whitelist_errors)}"
assert len(math_format_errors) == 0, f"Remaining math format errors found: {len(math_format_errors)}"
assert len(source_id_duplicates) == 0, f"Duplicate source_id found: {len(source_id_duplicates)}"

print("PASS: wiki_mcq_seed_final.jsonl passed all final checks.")

AssertionError: Remaining math format errors found: 29

### 29 lỗi -> sửa tay cho nhanh